# Spectral clustering

_Classical ML_

**Cluster by connectivity, not distance — cut the graph at its thin seams.**

Two crescents can interleave so that points across the gap are closer than points along the same arc.

     k-means, which judges by raw distance, fails badly here.

---

This notebook builds spectral clustering up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We build the idea on a clean toy problem first: two interleaving half-moons. We compare spectral clustering against k-means in three steps — (1) make the data, (2) cluster it both ways, (3) score who recovered the true moons.

### Step 1 — Make two interleaving half-moons

`make_moons` draws two crescent shapes that curl around each other. Points on opposite arcs can sit closer together than points on the same arc, so plain distance is a misleading guide to cluster membership. A little `noise` blurs the arcs to make the task realistic.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

# Two crescents of 300 points; noise blurs the arcs slightly.
X, y = make_moons(n_samples=300, noise=0.06, random_state=0)

### Step 2 — Cluster two ways: spectral vs k-means

Spectral clustering builds a **nearest-neighbours graph** (each point links to its 10 closest neighbours), then cuts that graph at its thin seams — so connectivity, not raw distance, decides the groups. k-means instead carves space into two round blobs around their centres. We run both and keep the predicted labels.

In [ ]:
from sklearn.cluster import SpectralClustering, KMeans

# Spectral: cluster by graph connectivity (nearest-neighbour affinity).
sc = SpectralClustering(n_clusters=2, affinity="nearest_neighbors",
                        n_neighbors=10, assign_labels="kmeans",
                        random_state=0)
sc_labels = sc.fit_predict(X)

# k-means: cluster by distance to two centroids.
km = KMeans(n_clusters=2, n_init=10, random_state=0)
km_labels = km.fit_predict(X)

### Step 3 — Score who recovered the moons

The **adjusted Rand index (ARI)** compares predicted clusters to the true labels: `1.0` is a perfect match, `0.0` is chance. Spectral clustering follows the curved arcs and scores near 1; k-means, forced into round blobs, slices each moon in half and scores much lower.

In [ ]:
from sklearn.metrics import adjusted_rand_score

print("spectral cluster sizes:", np.bincount(sc_labels))
print("spectral ARI vs truth:", round(adjusted_rand_score(y, sc_labels), 3))
print("k-means  ARI vs truth:", round(adjusted_rand_score(y, km_labels), 3))
print("-> spectral follows the curved moons; k-means does not")

## Visualize the data & results

_On the real Wine cultivars, does spectral clustering recover the same groups as k-means?_ We move from the toy moons to a real 13-feature dataset, in three steps: (1) prepare the data in 2D, (2) cluster it both ways, (3) plot the two results side by side.

### Step 1 — Standardise the Wine data and project to 2D

The Wine dataset has 13 chemical measurements on three cultivars. We **standardise** every feature (so no single column dominates by scale), then use **PCA** to compress the 13 dimensions down to the 2 that capture the most variation — giving us coordinates we can actually plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

wine = load_wine()

# Put every feature on the same scale before clustering.
Xs = StandardScaler().fit_transform(wine.data)

# Compress 13 features down to the 2 most informative directions.
X2 = PCA(n_components=2, random_state=0).fit_transform(Xs)

### Step 2 — Cluster the projected data both ways

Now we ask each method for **three** clusters (one per cultivar) on the 2D points. We print the ARI of each against the true cultivar labels so we can see, numerically, which method aligns better with reality on this dataset.

In [ ]:
from sklearn.cluster import SpectralClustering, KMeans
from sklearn.metrics import adjusted_rand_score

sc = SpectralClustering(n_clusters=3, affinity="nearest_neighbors",
                        n_neighbors=10, random_state=0).fit_predict(X2)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X2)

print(adjusted_rand_score(wine.target, sc),
      adjusted_rand_score(wine.target, km))

### Step 3 — Plot the two clusterings side by side

We colour each point by its assigned cluster, with spectral on the left and k-means on the right. Here the cultivars form roughly round, well-separated blobs in PCA space — exactly the case k-means is built for — so the two pictures look similar, unlike the moons.

In [ ]:
colors = np.array(["#4ea1ff", "#7ee787", "#c89bff"])

fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, sharey=True)

ax1.scatter(X2[:, 0], X2[:, 1], c=colors[sc], s=18)
ax1.set_title("Spectral clustering on Wine")

ax2.scatter(X2[:, 0], X2[:, 1], c=colors[km], s=18)
ax2.set_title("k-means on Wine")

for ax in (ax1, ax2):
    ax.set_xlabel("PCA component 1")
    ax.set_ylabel("PCA component 2")

plt.show()